# 🧠 LM Notebook: O Cérebro do seu Projeto com Memória MemPalace

Bem-vindo ao **LM Notebook**, seu assistente cognitivo e de pesquisa local! Este notebook serve como interface do seu "cérebro" de IA integrado ao sistema de memória persistente **MemPalace**.

### 🏛️ Como funciona a Memória?
O MemPalace armazena todo o histórico de conversas e documentos de forma literal (verbatim), estruturando-os na metáfora do palácio:
*   **Wings (Alas):** Seus projetos (ex: `TR-Project`).
*   **Rooms (Salas):** Tópicos de discussão (ex: `design_system`, `business_rules`).
*   **Drawers (Gavetas):** Conteúdo detalhado das memórias e arquivos.

Utilizando busca semântica em um banco de dados vetorial (**ChromaDB**), a IA recupera o contexto exato e responde perguntas baseadas em seus próprios documentos, funcionando exatamente como o **Google NotebookLM**, porém **totalmente local e privado**!

---  
## 🛠️ Passo 1: Instalação e Preparação do Ambiente

Primeiro, vamos garantir que todas as dependências estão instaladas no seu ambiente. Execute a célula abaixo para instalar o `mempalace` e os clientes de API necessários.

In [ ]:
# Instalação dos pacotes recomendados
!pip install mempalace openai google-generativeai chromadb ipywidgets

---  
## 🏛️ Passo 2: Inicializando o Palácio de Memórias

Precisamos inicializar o diretório de dados do MemPalace. Por padrão, ele cria a base em `~/.mempalace/` no seu computador.  
Rode a célula a seguir para executar a inicialização.

In [ ]:
import os
import subprocess

# Executa a inicialização do MemPalace
try:
    result = subprocess.run(["mempalace", "init"], capture_output=True, text=True, check=True)
    print("✅ MemPalace inicializado com sucesso!")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("⚠️ Nota: MemPalace já pode estar inicializado ou ocorreu uma resposta:")
    print(e.stderr or e.stdout)
except FileNotFoundError:
    print("❌ Erro: O comando 'mempalace' não foi encontrado. Certifique-se de que o pip install terminou com sucesso e o ambiente está ativo.")

---  
## 📂 Passo 3: Mineração de Dados (Ingestão de Arquivos - Estilo NotebookLM)

Vamos fazer o cérebro da IA ler a pasta `test_memory/` onde salvamos o arquivo `brain_instructions.txt` (que contém informações de regras de negócio, cores de design e o segredo `CRISTAL-AZUL`).

Este comando varre os arquivos e cria os embeddings vetoriais na base semântica local.

In [ ]:
import os
import subprocess

# Pasta de memória do projeto
target_dir = os.path.abspath("./test_memory")

print(f"Minerando documentos da pasta: {target_dir}")
try:
    result = subprocess.run(["mempalace", "mine", target_dir], capture_output=True, text=True, check=True)
    print("🎉 Ingestão concluída com sucesso! Os arquivos foram memorizados pelo Cérebro.")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("❌ Erro ao minerar pasta:")
    print(e.stderr or e.stdout)

---  
## 🔎 Passo 4: Busca Semântica Programática

Agora podemos pesquisar diretamente nas memórias usando a API do MemPalace! Vamos testar procurando pela nossa palavra-chave secreta `CRISTAL-AZUL` ou pelas cores do design.

In [ ]:
try:
    from mempalace.searcher import search_memories
    
    # Executa a pesquisa
    query = "Qual é a palavra-chave secreta do projeto?"
    print(f"Pesquisando por: '{query}'...\n")
    
    results = search_memories(query)
    
    if not results:
        print("ℹ️ Nenhuma memória correspondente encontrada. Tente minerar mais arquivos primeiro.")
    else:
        for idx, mem in enumerate(results, 1):
            print(f"📍 Memória #{idx} (Ala: {mem.get('wing', 'N/A')}, Sala: {mem.get('room', 'N/A')}):")
            print(f"   {mem.get('content', '').strip()}")
            print("-" * 50)
except ImportError:
    # Fallback se o import direto tiver diferenças na versão instalada
    print("⚠️ Import direto indisponível. Rodando busca via CLI de backup:\n")
    import subprocess
    res = subprocess.run(["mempalace", "search", "CRISTAL-AZUL"], capture_output=True, text=True)
    print(res.stdout)

---  
## 🧠 Passo 5: Construindo o Cérebro do Assistente (Chat Interativo)

Aqui unimos a memória semântica com o modelo de linguagem (LLM). Você pode configurar abaixo para rodar com **LM Studio local**, **Ollama**, ou APIs como **Google Gemini** ou **OpenAI**.

In [ ]:
import os
from openai import OpenAI
import google.generativeai as genai

# ================= CONFIGURAÇÃO DO PROVEDOR DE LLM =================
# Defina qual provedor de LLM deseja usar: 'lm_studio', 'ollama', 'gemini' ou 'openai'
PROVIDER = 'lm_studio'

# Configuração das chaves API (se usar nuvem)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "SUA_CHAVE_GEMINI_AQUI")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "SUA_CHAVE_OPENAI_AQUI")

# Porta padrão do LM Studio e Ollama
LM_STUDIO_URL = "http://localhost:1234/v1"
OLLAMA_URL = "http://localhost:11434/v1"
# ===================================================================

def call_llm(system_prompt, user_query):
    """Invoca o modelo de linguagem de acordo com o provedor escolhido."""
    if PROVIDER == 'lm_studio':
        client = OpenAI(base_url=LM_STUDIO_URL, api_key="not-needed")
        response = client.chat.completions.create(
            model="local-model", # LM studio seleciona o modelo ativo automaticamente
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content
        
    elif PROVIDER == 'ollama':
        client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")
        response = client.chat.completions.create(
            model="llama3", # Ajuste para o modelo instalado no seu Ollama
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ]
        )
        return response.choices[0].message.content
        
    elif PROVIDER == 'gemini':
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-1.5-flash')
        combined_prompt = f"{system_prompt}\n\nUsuário: {user_query}"
        response = model.generate_content(combined_prompt)
        return response.text
        
    elif PROVIDER == 'openai':
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ]
        )
        return response.choices[0].message.content
    else:
        return "⚠️ Provedor de LLM inválido configurado."

print(f"🧠 Cérebro pronto usando o provedor: '{PROVIDER}'")

### 🤖 Função de Chat Inteligente (com Ingestão Semântica)

Esta função pesquisa no banco do MemPalace antes de enviar a pergunta à IA. Ela injeta as memórias relevantes no prompt do sistema como referências contextualizadas.

In [ ]:
from mempalace.searcher import search_memories

def chat_with_brain(question):
    print(f"❓ Pergunta do Usuário: {question}\n")
    
    # 1. Recupera contexto relevante da memória do MemPalace
    print("🔍 Consultando a memória de longo prazo (MemPalace)...")
    try:
        memories = search_memories(question)
    except Exception:
        memories = []
        
    context_blocks = []
    citations = []
    
    for idx, mem in enumerate(memories, 1):
        content = mem.get('content', '').strip()
        wing = mem.get('wing', 'N/A')
        room = mem.get('room', 'N/A')
        context_blocks.append(f"[Memória #{idx} | Ala: {wing}, Sala: {room}]:\n{content}")
        citations.append(f"- Memória #{idx} (Ala: {wing}, Sala: {room})")
        
    context = "\n\n".join(context_blocks) if context_blocks else "Nenhuma memória relevante encontrada no banco."
    
    # 2. Constrói o Prompt de Sistema estruturado (Grounded AI)
    system_prompt = f"""Você é o cérebro cognitivo do projeto TR, auxiliando o usuário com base nas memórias e documentos fornecidos.
Sua prioridade máxima é responder de forma precisa fundamentando-se nas memórias injetadas abaixo.

--- MEMÓRIAS RECUPERADAS DO PALÁCIO ---
{context}
-----------------------------------------

Caso o usuário faça uma pergunta sobre algo que esteja contido nessas memórias, cite as fontes ou o número da memória em sua resposta.
Se as memórias não possuírem a informação, use seu conhecimento geral de forma clara e informe de forma transparente que a informação não veio da memória persistente.
"""
    
    # 3. Invoca o modelo
    print("🧠 Pensando...")
    try:
        answer = call_llm(system_prompt, question)
        print("\n✨ RESPOSTA DO CÉREBRO:")
        print(answer)
        if citations:
            print("\n📚 Fontes Consultadas (Citações estilo NotebookLM):")
            for cit in citations:
                print(cit)
    except Exception as e:
        print(f"\n❌ Erro ao invocar a LLM: {str(e)}")
        print("Certifique-se de que o provedor escolhido está rodando ou configurado corretamente com a chave API!")

### 💬 Teste o Cérebro Agora!

Experimente rodar a célula abaixo fazendo perguntas que testem as memórias do arquivo minerado (ex: "Qual é o segredo do projeto?" ou "Quais são as cores do design?").

In [ ]:
# Rode esta célula com sua pergunta!
chat_with_brain("Qual é a palavra-chave secreta do projeto?")

---  
## 💾 Passo 6: Persistindo Novas Memórias Dinamicamente

Deseja ensinar algo novo para a IA e fazer ela lembrar para sempre?  
Criamos uma função que escreve o conhecimento em um arquivo e o envia diretamente para o banco de dados vetorial do MemPalace!

In [ ]:
import os
import subprocess

def teach_new_memory(room, title, content):
    """
    Cria um arquivo temporário de conhecimento e o minera no MemPalace.
    """
    temp_dir = os.path.abspath("./temp_memories")
    os.makedirs(temp_dir, exist_ok=True)
    
    filename = f"{room}_{title.replace(' ', '_').lower()}.txt"
    filepath = os.path.join(temp_dir, filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Sala/Tópico: {room}\nTítulo: {title}\n\n{content}")
        
    print(f"💾 Criado arquivo temporário em: {filepath}")
    
    # Executa a mineração
    try:
        subprocess.run(["mempalace", "mine", temp_dir], capture_output=True, text=True, check=True)
        print(f"🎉 O cérebro gravou com sucesso o conhecimento '{title}' na sala '{room}'!")
        
        # Remove arquivo temporário após mineração bem-sucedida para manter a pasta limpa
        os.remove(filepath)
    except Exception as e:
        print(f"❌ Erro ao persistir a memória: {str(e)}")

# Exemplo de uso:
teach_new_memory(
    room="arquitetura", 
    title="Regra de Banco de Dados", 
    content="Todos os backups de produção devem ser executados diariamente às 03:00 AM no horário UTC."
)

### 🔄 Testando a nova memória adicionada

Rode a célula abaixo para comprovar que a IA acabou de aprender essa nova regra de banco!

In [ ]:
chat_with_brain("A que horas devem ser executados os backups de produção?")